<a id="nb06-top"></a>

# Notebook 06 — Physiographic Modelling of Urban Thermal Hotspots and Cold Spots

We test how the physiographic landscape of Nigeria relates to the surface hot and cold spots of the 56
study cities: whether the composite typology (Cluster25) tells cities apart, and which of its elements —
climate zone, geology, elevation — carries the signal.

- **Units:** 56 cities, and 185 spots (95 hot, 90 cold).
- **Data:** NB05's spot layer (`all_cities_spots.geojson`) and NB04's SUHII.

---

## Study Design Overview

| Item | Research Question | Analytical Strategy |
|------|-------------------|---------------------|
| **1 — Element-level attribution** | Do the individual landscape elements (climate zone, geology, elevation) relate to the spots? | Part B — chi-square / Kruskal-Wallis per element |
| **2 — Cluster-level validation** | Does the composite Cluster25 typology differentiate the cities' heating and cooling patterns? | Part A — chi-square / Kruskal-Wallis on Cluster25 |

We run **Part A** (the composite) first, to see whether the typology discriminates at all, then **Part B**
(the elements), to see which dimension drives it. The **synthesis** links the two (Cramér's V between
typology and elements, comparative explanatory power, the clusters' thermal profiles), and the
**appendix** holds the full charts. We analyse hot and cold spots, and day and night, separately throughout.

## Table of Contents

1. [§1 Setup](#nb06-s1)
2. [§2 Data Preparation](#nb06-s2)
3. [§3 Part A — Cluster-level Association](#nb06-s3)
4. [§4 Part B — Element-level Attribution](#nb06-s4)
5. [§5 Synthesis](#nb06-s5)
6. [§6 Appendix — Statistical Charts](#nb06-s6)
7. [§7 Master Synthesis Table and Forest Plot](#nb06-s7)
8. [§8 Notebook Summary and Key Outputs](#nb06-s8)

<a id="nb06-s1"></a>

## §1 — Setup and Imports

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, json
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import kruskal, chi2_contingency
import scikit_posthocs as sp

# ── Path setup ────────────────────────────────────────────────────────────────
ROOT = Path(os.getcwd())
if not (ROOT / 'config.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

NB05_RESULTS = ROOT / 'Results' / '05_hotspot_analysis'
RESULTS_DIR  = ROOT / 'Results' / '06_hotspot_modelling'
FIGS_DIR     = RESULTS_DIR / 'figures'
TABLES_DIR   = RESULTS_DIR / 'tables'


# ── Analysis constants ────────────────────────────────────────────────────────
ALPHA          = 0.05
PERIODS        = ['day', 'night']
SPOT_TYPES     = ['hot', 'cold']
ELEMENT_COLS   = ['climate_class', 'geology_class', 'elevation_class']
ELEMENT_LABELS = {
    'climate_class':   'Climate Zone',
    'geology_class':   'Geology',
    'elevation_class': 'Elevation Belt',
}
# Cluster25 colour palette — loaded from config to avoid duplication
_CLUST_PALETTE = config.CLUSTER25_COLORS

# ── City selection — mirrors config.CITY_SELECTION_MODE ─────────────────────
_sel_mode     = getattr(config, 'CITY_SELECTION_MODE', 'ABOVE_300K')
TARGET_CITIES = (list(config.TEST_CITIES)
                 if _sel_mode in ('TEST', 'SELECTED')
                 else None)   # None = all SEVIRI cities

MIN_CITIES_FOR_TEST = 10   # minimum cities needed for meaningful tests

if TARGET_CITIES is not None:
    _n = len(TARGET_CITIES)
    print(f'⚠ City selection mode: {_sel_mode!r}  ({_n} cities: {TARGET_CITIES})')
    if _n < MIN_CITIES_FOR_TEST:
        print(f'  ⚠ Statistical power warning: {_n} cities is below the recommended'
              f' minimum of {MIN_CITIES_FOR_TEST} for chi-square and Kruskal-Wallis.')
        print(f'  Tests will still run, but results should be interpreted with caution.')
        print(f'  With {_n} cities, group sizes per cluster may be 1-2 observations,')
        print(f'  making effect size estimates unreliable. Consider using ABOVE_300K mode')
        print(f'  for a full-population analysis.')
    print()
else:
    print(f'City selection mode: {_sel_mode!r} — using all SEVIRI cities')


print(f"ROOT        : {ROOT}")
print(f"NB05 results: {NB05_RESULTS}")
print(f"NB06 results: {RESULTS_DIR}")
print(f"scikit_posthocs loaded ✓")

<a id="nb06-s2"></a>

## §2 — Data Preparation

### §2.1 Load NB05 GDF

We load NB05's spot layer (`all_cities_spots.geojson`, from NB05 §7):

| Column group | Key columns | Notes |
|---|---|---|
| Identity | `city`, `period`, `spot_type`, `spot_id` | Primary keys |
| Geometry | `area_km2_50`, `area_km2_75`, `area_km2_90` | Contour zone areas (km²) |
| Landscape | `cluster25`, `climate_class`, `geology_class`, `elevation_class` | City-level dominant physical class via AOI area-weighted intersection |
| Thermal | `suhii` *(primary)*, `mean_lst`, `core_mean_lst` *(reference)* | `suhii` = city-level SUHII from NB04; spatial median UC − BRA (same value for all spots in city × period) |
| Population | `population`, `pop_75`, `pop_90` | GHSL population within the contour zones |
The landscape classes are **city-level**: each city's dominant class, by area-weighted overlay of its AOI on
the typology polygons, shared by all its spots.

**SUHII definition:**

$$\text{SUHII}_{c,p} = \widetilde{\text{LST}}_{\text{UC},c,p} \;-\; \widetilde{\text{LST}}_{\text{BRA},c,p}$$
with $c$ the city, $p$ the period and $\widetilde{\text{LST}}$ the spatial median LST of the zone: the urban
core (UC, the SMOD built-up extent) and the background reference area (BRA, a 10 km ring without water and
urban pixels).

In [ ]:
# ── Load GDF ─────────────────────────────────────────────────────────────────
gdf_path = NB05_RESULTS / 'all_cities_spots.geojson'
gdf = gpd.read_file(gdf_path)

# Apply city filter if TARGET_CITIES is set
if TARGET_CITIES is not None:
    gdf = gdf[gdf['city'].isin(TARGET_CITIES)].copy()
    print(f'GDF filtered to {len(TARGET_CITIES)} target cities: {len(gdf)} spots')

for col in ('bh_significant', 'persistent_dn'):
    if col in gdf.columns:
        gdf[col] = gdf[col].astype(bool)

# Ensure numeric types
for col in ['area_km2_50', 'area_km2_90', 'suhii', 'n_pixels']:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce')

print(f"GDF: {len(gdf)} spots × {len(gdf.columns)} columns")
print(f"  Hot  : {(gdf.spot_type=='hot').sum()} spots")
print(f"  Cold : {(gdf.spot_type=='cold').sum()} spots")
print(f"  Cities in GDF: {gdf.city.nunique()}")
print(f"\nBreakdown by period x spot_type:")
print(gdf.groupby(['period','spot_type']).agg(
    n_spots=('spot_id','count'), n_cities=('city','nunique'),
    median_suhii=('suhii','median'), median_area_km2_50=('area_km2_50','median')
).round(3).to_string())
print(f"\nLandscape class distributions:")
for col in ELEMENT_COLS:
    vals = gdf[col].value_counts()
    print(f"  {ELEMENT_LABELS[col]}: {vals.to_dict()}")

### §2.2 City Landscape Class Lookup

Each city has one dominant class per element. For cities with spots we read it from their spot rows; for
cities without spots we derive it in the same way, by area-weighted overlay of the city AOI $A_c$ with the
typology polygons $\{L_j\}$. For each element $d \in \{\text{climate},\,\text{geology},\,\text{elevation}\}$:

$$k^* = \underset{k}{\arg\max} \sum_{j:\, L_j^{(d)} = k} \text{Area}(L_j \cap A_c)$$

so that the class reflects the city's physical footprint. We leave land cover out: it partly reflects urban
change rather than the natural setting.

In [ ]:
# ── City landscape class lookup from GDF ─────────────────────────────────────
city_landscape = (
    gdf[['city', 'cluster25', 'climate_class', 'geology_class',
          'elevation_class', 'pct_dominant_class']]
    .drop_duplicates('city')
    .reset_index(drop=True)
)

# ── All cities with SEVIRI coverage ──────────────────────────────────────────
seviri_dir  = ROOT / 'Results' / '04_temporal_analysis_daily' / 'seviri'
all_cities  = sorted([d.name for d in seviri_dir.iterdir()
                      if d.is_dir() and d.name != 'mosaics'])
if TARGET_CITIES is not None:
    all_cities = [c for c in all_cities if c in TARGET_CITIES]
gdf_cities  = set(city_landscape['city'])
no_spot_cities = [c for c in all_cities if c not in gdf_cities]

print(f"All SEVIRI cities   : {len(all_cities)}")
print(f"Cities with spots   : {len(gdf_cities)}")
print(f"Cities without spots: {no_spot_cities or 'None'}")

# ── Landscape classes for no-spot cities ──────────────────────────────────────
# Taken from the published city profile (v1.0.1 package, layer city_profile): the same
# assignment the GDF carries for every city with spots, asserted below. (The earlier AOI
# lookup read the old folder layout City_<city>/Features/, which no longer exists; it
# failed, the no-spot city (Ikare) kept no cluster, and dropna() removed it from every
# occurrence test: 55 cities instead of 56.)
import pyogrio
_pkg_gpkg = (ROOT / 'Results' / '05_hotspot_analysis'
             / 'surface_thermal_anomaly_stakeholder_composite'
             / 'surface_thermal_anomaly_stakeholder_composite.gpkg')
_cp = pyogrio.read_dataframe(_pkg_gpkg, layer='city_profile', read_geometry=False)
_cp = _cp[['city', 'cluster25'] + ELEMENT_COLS + ['pct_dominant_class']].copy()
_cp['cluster25'] = _cp['cluster25'].astype(int)
_chk = city_landscape.merge(_cp, on='city', suffixes=('', '_pkg'))
assert len(_chk) == len(city_landscape) and all(
    (_chk[c].astype(str) == _chk[f'{c}_pkg'].astype(str)).all()
    for c in ['cluster25'] + ELEMENT_COLS), 'GDF landscape classes differ from city_profile'
if no_spot_cities:
    city_landscape = pd.concat(
        [city_landscape, _cp[_cp['city'].isin(no_spot_cities)][city_landscape.columns]],
        ignore_index=True)
    print(f"  Landscape classes from city_profile for: {no_spot_cities}")
assert sorted(city_landscape['city']) == sorted(all_cities), 'a city has no landscape class'
assert city_landscape[['cluster25'] + ELEMENT_COLS].notna().all().all(), 'a landscape class is missing'

print(f"\nCity landscape table: {len(city_landscape)} cities")
print(city_landscape[ELEMENT_COLS + ['cluster25']].value_counts(
    ELEMENT_COLS).rename('n_cities').to_frame().to_string())

### §2.3 City-Level Analysis Tables

For each period × spot type we build one table per city:

| Column | Definition | Coverage |
|---|---|---|
| `occurrence` | 1 = city has ≥1 validated spot for this period/type; 0 = none | All 56 cities |
| `n_spots` | Count of validated spot polygons | Cities with occurrence = 1 |
| `total_area_km2_50` | Sum of L50 contour areas (km²) across all spots in city — **primary area metric** (100% coverage, includes Tier-2 spots) | Cities with occurrence = 1 |
| `total_area_km2_90` | Sum of L90 (dense-core) contour areas (km²) — retained for reference (67% coverage; excludes all Tier-2 spots) | Cities with occurrence = 1 |
| `suhii` | City-level SUHII (°C) from NB04, same for all spots in city × period | Cities with occurrence = 1 |
| `cluster25`, `climate_class`, … | Dominant landscape class for city AOI | All 56 cities |
The SUHII and area tests use only the cities with spots (a city without one has no anomaly to measure);
the occurrence test uses all 56 cities.

In [ ]:
def build_city_table(gdf, all_cities, city_landscape, spot_type, period):
    # Build city-level table (all 56 cities) for a given period x spot_type.
    sub = gdf[(gdf.spot_type == spot_type) & (gdf.period == period)].copy()

    agg = (sub.groupby('city')
              .agg(n_spots        = ('spot_id',      'count'),
                   total_area_90  = ('area_km2_90',  'sum'),
                   total_area_50  = ('area_km2_50',  'sum'),
                   suhii          = ('suhii',        'first'))
              .reset_index()
              .rename(columns={'total_area_90': 'total_area_km2_90',
                               'total_area_50': 'total_area_km2_50'
                               }))
    agg['occurrence'] = 1

    # Cities without spots for this period/type
    cities_without = [c for c in all_cities if c not in set(agg['city'])]
    if cities_without:
        _no = pd.DataFrame({
            'city':              cities_without,
            'n_spots':           0,
            'total_area_km2_90': 0.0,
            'total_area_km2_50': 0.0,
            'suhii':             np.nan,
            'occurrence':        0,
        })
        agg = pd.concat([agg, _no], ignore_index=True)

    agg = agg.merge(city_landscape, on='city', how='left')
    agg['spot_type'] = spot_type
    agg['period']    = period
    return agg.sort_values('city').reset_index(drop=True)


hot_day    = build_city_table(gdf, all_cities, city_landscape, 'hot',  'day')
hot_night  = build_city_table(gdf, all_cities, city_landscape, 'hot',  'night')
cold_day   = build_city_table(gdf, all_cities, city_landscape, 'cold', 'day')
cold_night = build_city_table(gdf, all_cities, city_landscape, 'cold', 'night')

CITY_TABLES = {
    ('hot',  'day'):   hot_day,
    ('hot',  'night'): hot_night,
    ('cold', 'day'):   cold_day,
    ('cold', 'night'): cold_night,
}

print("City-level tables built:")
hdr = f"{'Table':<18} {'N':>4} {'Occurrence=1':>13} {'Occurrence=0':>13} "
hdr += f"{'Median SUHII':>14} {'Median Area_50':>15}"
print(hdr)
print('-' * len(hdr))
for (st, per), df in CITY_TABLES.items():
    n1 = (df.occurrence == 1).sum()
    n0 = (df.occurrence == 0).sum()
    med_suhii = df[df.occurrence == 1]['suhii'].median()
    med_area  = df[df.occurrence == 1]['total_area_km2_50'].median()
    print(f"{st+'-'+per:<18} {len(df):>4} {n1:>13} {n0:>13} "
          f"{med_suhii:>14.3f} {med_area:>15.3f}")

<a id="nb06-s3"></a>

## §3 — Part A: Cluster-level Association (Item 2)

### Analytical Rationale

Part A asks: *does the Cluster25 typology explain how the cities' surface hot and cold spots vary?*
Cluster25 is the physiographic landscape typology of Nigeria, each class a combination of climate,
geology and elevation. If cities of one cluster behave alike, the typology is a useful lens on heating and
cooling. SUHII is the main outcome; spot area adds the spatial dimension. We run three tests:

| Test | Outcome | Scope | Method |
|---|---|---|---|
| A1 | Occurrence | All 56 cities | Chi-square (Monte Carlo permutation p) + Cramér's V |
| A2 | **SUHII** *(primary)* | Cities with spots only | Kruskal-Wallis + Dunn post-hoc |
| A3 | Total area L50 *(reference)* | Cities with spots only | Kruskal-Wallis + Dunn post-hoc |

---

### Statistical Methods

#### A1 — Chi-square Test of Independence

We test whether the chance of a city having a hot (or cold) spot differs between clusters:

$$\chi^2 = \sum_{i=1}^{r} \sum_{j=1}^{c} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \quad E_{ij} = \frac{R_i \cdot C_j}{N}$$

| Symbol | Meaning |
|---|---|
| $O_{ij}$ | Observed count in row $i$, column $j$ |
| $E_{ij}$ | Expected count under independence |
| $R_i, C_j$ | Row and column marginal totals |
| $N$ | Total number of cities |
| $df = (r-1)(c-1)$ | Degrees of freedom |
Because many expected counts are small, we take the p-value from a Monte Carlo permutation of the outcome
rather than from the asymptotic distribution.

**Effect size — Cramér's V:**

$$V = \sqrt{\frac{\chi^2 / N}{\min(r-1,\; c-1)}}$$

$V \in [0,1]$: conventions — small $\approx 0.10$, medium $\approx 0.30$, large $\approx 0.50$.
#### A2/A3 — Kruskal-Wallis One-Way Test

The rank-based counterpart of one-way ANOVA: do $k$ groups differ? We use it because SUHII and area are
right-skewed and the groups are small and unequal.

$$H = \frac{12}{N(N+1)} \sum_{j=1}^{k} \frac{R_j^2}{n_j} - 3(N+1)$$

| Symbol | Meaning |
|---|---|
| $N$ | Total observations across all groups |
| $k$ | Number of Cluster25 groups |
| $n_j$ | Number of observations in group $j$ |
| $R_j$ | Sum of ranks in group $j$ |

Under $H_0$ (all groups have identical distributions), $H \sim \chi^2_{k-1}$.
**Effect size — Eta-squared:**

$$\eta^2_H = \frac{H - k + 1}{N - k}$$

Interpretation: proportion of variance in ranks explained by group membership. Conventions — small $\approx 0.01$, medium $\approx 0.06$, large $\approx 0.14$.
#### Post-hoc — Dunn's Test (Holm-corrected)

When $H$ is significant, Dunn's test shows which cluster pairs differ:

$$z_{ij} = \frac{\bar{R}_i - \bar{R}_j}{\sigma_{ij}}, \quad \sigma_{ij} = \sqrt{\frac{N(N+1)}{12}\left(\frac{1}{n_i} + \frac{1}{n_j}\right)}$$
with the p-values Holm-corrected across all pairs (family-wise error rate).

<a id="nb06-s3-0"></a>

### §3.0 — Normality Diagnostics (Assumption Check)

We check the normality that one-way ANOVA would need, for SUHII and total L50 area in each spot type ×
period: Shapiro–Wilk (the main test), D'Agostino–Pearson K² (from n ≥ 8) and the skewness and excess
kurtosis. An outcome rejected as normal (p < α) or strongly skewed (|skew| > 1) needs a rank-based test;
where normality is not rejected, the small, unequal cluster groups still favour one, so we use Kruskal–Wallis
throughout (§3b, §3c). Occurrence (§3a) is binary and goes to the chi-square test.

**Output:** `normality_diagnostics.csv`

In [ ]:
# ── §3.0 — Normality diagnostics (assumption check for §3 Part A) ──────────────
# Shapiro-Wilk (primary; valid 3 ≤ n ≤ 5000) with D'Agostino-Pearson K² as a
# large-n cross-check, on the continuous outcomes fed to the Kruskal-Wallis tests
# (SUHII and total L50 area), for each spot_type × period. Skewness/kurtosis give
# the direction and heaviness of departure. Non-normal (p < ALPHA) or |skew| > 1
# is the empirical justification for the rank-based Kruskal-Wallis (§3b/§3c) over
# one-way ANOVA. Occurrence (§3a) is categorical → chi-square, no normality needed.
from scipy.stats import shapiro, normaltest, skew, kurtosis

_NORM_OUTCOMES = [('suhii', 'SUHII (°C)'),
                  ('total_area_km2_50', 'Total area L50 (km²)')]
_norm_rows = []
for (st, per), df in CITY_TABLES.items():
    _sub = df[df['occurrence'] == 1]          # cities with a spot = what KW uses
    for col, label in _NORM_OUTCOMES:
        vals = pd.to_numeric(_sub[col], errors='coerce').dropna().values
        n = int(len(vals))
        row = {'Outcome': label, 'Spot': st, 'Period': per, 'N': n,
               'Shapiro_W': np.nan, 'Shapiro_p': np.nan, 'DAgostino_p': np.nan,
               'Skew': np.nan, 'Excess_kurt': np.nan, 'Normal_at_alpha': None}
        if n >= 3 and np.ptp(vals) > 0:
            W, p_sw = shapiro(vals)
            row['Shapiro_W'] = round(float(W), 4)
            row['Shapiro_p'] = float(p_sw)
            if n >= 8:                          # D'Agostino needs n ≥ 8
                row['DAgostino_p'] = float(normaltest(vals).pvalue)
            row['Skew'] = round(float(skew(vals)), 3)
            row['Excess_kurt'] = round(float(kurtosis(vals)), 3)
            row['Normal_at_alpha'] = bool(p_sw >= ALPHA)
        _norm_rows.append(row)

normality_df = pd.DataFrame(_norm_rows)

def _pn(p):
    return '   n/a' if pd.isna(p) else ('<0.001' if p < 0.001 else f'{p:.3f}')

print('NORMALITY DIAGNOSTICS — continuous Kruskal-Wallis outcomes (cities with spots)')
print('=' * 82)
print(f"{'Outcome':22s} {'Spot':5s} {'Period':6s} {'N':>3s} {'Shapiro W':>10s} "
      f"{'Shapiro p':>10s} {'DAgo p':>8s} {'Skew':>7s} {'Verdict':>11s}")
print('-' * 82)
for _, r in normality_df.iterrows():
    verdict = ('—' if r['Normal_at_alpha'] is None
               else ('Normal' if r['Normal_at_alpha'] else 'NON-normal'))
    _w  = f"{r['Shapiro_W']:.4f}" if pd.notna(r['Shapiro_W']) else '   n/a'
    _sk = f"{r['Skew']:+.2f}"     if pd.notna(r['Skew'])      else '   n/a'
    print(f"{r['Outcome']:22s} {r['Spot']:5s} {r['Period']:6s} {int(r['N']):>3d} "
          f"{_w:>10s} {_pn(r['Shapiro_p']):>10s} {_pn(r['DAgostino_p']):>8s} "
          f"{_sk:>7s} {verdict:>11s}")
print('-' * 82)
_tested   = int(normality_df['Shapiro_p'].notna().sum())
_nonnorm  = int((normality_df['Shapiro_p'] < ALPHA).sum())
print(f"Non-normal at α={ALPHA} (Shapiro): {_nonnorm}/{_tested} distributions.")

_norm_out = RESULTS_DIR / 'normality_diagnostics.csv'
normality_df.to_csv(_norm_out, index=False)
print(f"  ✓ Saved: {_norm_out.name}")


In [ ]:
# ── Statistical helper functions ─────────────────────────────────────────────

def run_chi2(df, groupvar, outcome='occurrence', n_perm=9999, seed=42):
    """Monte Carlo permutation chi-square + Cramér's V.

    Uses permutation instead of the asymptotic chi-square distribution to avoid
    the low-expected-count assumption (conventional rule: ≤20% of cells with
    expected < 5).  Returns dict with both asymptotic and permutation p-values.
    """
    sub = df[[groupvar, outcome]].dropna()
    sub = sub[sub[groupvar] != -1] if groupvar == 'cluster25' else sub
    if sub.empty or sub[groupvar].nunique() < 2:
        return None
    ct = pd.crosstab(sub[groupvar], sub[outcome])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return None

    observed = ct.values.astype(float)
    chi2_obs, p_asymp, dof, expected = chi2_contingency(observed)
    n        = int(observed.sum())
    min_dim  = min(ct.shape[0] - 1, ct.shape[1] - 1)
    v        = np.sqrt(chi2_obs / (n * max(min_dim, 1)))
    low_exp  = int((expected < 5).sum())

    # ── Monte Carlo permutation ────────────────────────────────────────────────
    rng     = np.random.default_rng(seed)
    # Expand contingency table back to individual observations
    grp_arr = np.repeat(np.arange(observed.shape[0]),
                        observed.sum(axis=1).astype(int))
    out_arr = np.repeat(np.arange(observed.shape[1]),
                        observed.sum(axis=0).astype(int))

    count_ge = 0
    for _ in range(n_perm):
        perm_out   = rng.permutation(out_arr)
        ct_perm    = np.zeros_like(observed)
        np.add.at(ct_perm, (grp_arr, perm_out), 1)
        chi2_p, *_ = chi2_contingency(ct_perm)
        if chi2_p >= chi2_obs:
            count_ge += 1

    p_mc = (count_ge + 1) / (n_perm + 1)
    use_mc = low_exp > 0  # use MC p if any low-expected cells

    return {
        'chi2':              chi2_obs,
        'p':                 p_mc if use_mc else p_asymp,   # primary p-value
        'p_asymptotic':      p_asymp,
        'p_mc':              p_mc,
        'method':            f'MC permutation (n={n_perm})' if use_mc else 'asymptotic',
        'dof':               dof,
        'cramers_v':         v,
        'n':                 n,
        'n_groups':          ct.shape[0],
        'low_expected_cells': low_exp,
        'contingency_table': ct,
    }


def run_kw(df, groupvar, valvar, min_group=3):
    """Kruskal-Wallis + Holm-corrected Dunn post-hoc.

    Uses only cities with occurrence = 1. Skips groups smaller than min_group.
    Prints a statistical power warning if total N < MIN_CITIES_FOR_TEST.
    """
    sub = df[df['occurrence'] == 1][[groupvar, valvar]].dropna()
    if groupvar == 'cluster25':
        sub = sub[sub[groupvar] != -1]
    groups = {name: grp[valvar].values
              for name, grp in sub.groupby(groupvar)
              if len(grp) >= min_group}
    if len(groups) < 2:
        return None

    vals = list(groups.values())
    H, p = kruskal(*vals)
    k    = len(vals)
    N    = sum(len(v) for v in vals)
    eta2 = max(0.0, (H - k + 1) / (N - k)) if N > k else 0.0

    if N < MIN_CITIES_FOR_TEST:
        print(f"  ⚠ Power warning: {valvar} KW test  N={N} < {MIN_CITIES_FOR_TEST}. "
              f"Results should be interpreted cautiously.")

    try:
        dunn = sp.posthoc_dunn(
            sub[sub[groupvar].isin(groups.keys())],
            val_col=valvar, group_col=groupvar, p_adjust='holm'
        )
    except Exception:
        dunn = None

    return {
        'H': H, 'p': p, 'eta2': eta2, 'k': k, 'N': N,
        'group_medians': sub.groupby(groupvar)[valvar].median().to_dict(),
        'group_n':       sub.groupby(groupvar)[valvar].count().to_dict(),
        'dunn':          dunn,
    }


def sig_stars(p):
    if p is None or np.isnan(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def fmt_p(p):
    if p is None or np.isnan(p): return '—'
    if p < 0.001: return '< 0.001'
    return f'{p:.3f}'


print("Helper functions defined: run_chi2 (MC permutation), run_kw, sig_stars, fmt_p ✓")


### §3a — A1: Hotspot Occurrence × Cluster25

Does a city's cluster predict whether it has hot (or cold) spots? We test the binary outcome against the
active Cluster25 classes for hot-day, hot-night, cold-day and cold-night.

In [ ]:
# ── A1: Occurrence × Cluster25 ───────────────────────────────────────────────
partA_occ = {}

for (st, per), df in CITY_TABLES.items():
    result = run_chi2(df, 'cluster25', 'occurrence')
    partA_occ[(st, per)] = result
    if result:
        ct = result['contingency_table']
        print(f"\n{'='*60}")
        print(f"  {st.upper()} spots — {per.upper()} | Cluster25 × Occurrence")
        print(f"{'='*60}")
        print(f"  N cities : {result['n']}")
        print(f"  N clusters: {result['n_groups']}")
        print(f"  χ²({result['dof']}) = {result['chi2']:.3f},  p = {fmt_p(result['p'])} {sig_stars(result['p'])}")
        print(f"  Cramér's V = {result['cramers_v']:.3f}")
        if result['low_expected_cells'] > 0:
            print(f"  ⚠ {result['low_expected_cells']} cells with expected count < 5")
        print(f"\n  Contingency table (rows=Cluster25, cols=occurrence):")
        print(ct.to_string())
        pct = ct.div(ct.sum(axis=1), axis=0).mul(100).round(1)
        pct.columns = ['0%', '1%']
        print(f"\n  Row percentages (% with occurrence=1):")
        print(pct.to_string())
    else:
        print(f"\n  {st}-{per}: insufficient data for chi-square")

### §3b — A2: SUHII × Cluster25

Among the cities with spots, does their SUHII differ between clusters? SUHII (NB04) is the urban minus
background median LST: we expect it positive for hot-spot cities and negative (urban cooling) for
cold-spot cities. Only cities with `occurrence = 1` enter.

In [ ]:
# ── A2: SUHII × Cluster25 ────────────────────────────────────────────────────
partA_suhii = {}

for (st, per), df in CITY_TABLES.items():
    result = run_kw(df, 'cluster25', 'suhii')
    partA_suhii[(st, per)] = result
    if result:
        print(f"\n{'='*60}")
        print(f"  {st.upper()} spots — {per.upper()} | SUHII × Cluster25")
        print(f"{'='*60}")
        print(f"  N={result['N']}, k={result['k']} active clusters")
        print(f"  H = {result['H']:.3f},  p = {fmt_p(result['p'])} {sig_stars(result['p'])}")
        print(f"  η² = {result['eta2']:.4f}  ({result['eta2']*100:.1f}% variance explained)")
        print(f"\n  Cluster medians (SUHII °C):")
        for cl, med in sorted(result['group_medians'].items()):
            n = result['group_n'][cl]
            print(f"    Cluster {cl:>2}: {med:>+7.3f} °C  (n={n})")
        if result['p'] < ALPHA and result['dunn'] is not None:
            print(f"\n  → Significant: Dunn post-hoc results saved to partA_suhii[({st!r},{per!r})]['dunn']")
    else:
        print(f"\n  {st}-{per}: fewer than 2 clusters with ≥3 observations — skipped")

### §3c — A3: Total Hotspot Area × Cluster25

Among the cities with spots, does their total spot extent — the L50 contour areas summed over the city's
spots — differ between clusters? We use L50 because every spot has one, Tier-2 included, while L90 exists
for only 67 % of the spots. The areas are right-skewed (a few large cities dominate), which again favours
Kruskal–Wallis.

In [ ]:
# ── A3: Total hotspot area × Cluster25 ───────────────────────────────────────
partA_area = {}

for (st, per), df in CITY_TABLES.items():
    result = run_kw(df, 'cluster25', 'total_area_km2_50')
    partA_area[(st, per)] = result
    if result:
        print(f"\n{'='*60}")
        print(f"  {st.upper()} spots — {per.upper()} | Area (L50) × Cluster25")
        print(f"{'='*60}")
        print(f"  N={result['N']}, k={result['k']} active clusters")
        print(f"  H = {result['H']:.3f},  p = {fmt_p(result['p'])} {sig_stars(result['p'])}")
        print(f"  η² = {result['eta2']:.4f}  ({result['eta2']*100:.1f}% variance explained)")
        print(f"\n  Cluster medians (total area L50, km²):")
        for cl, med in sorted(result['group_medians'].items()):
            n = result['group_n'][cl]
            print(f"    Cluster {cl:>2}: {med:>8.3f} km²  (n={n})")
    else:
        print(f"\n  {st}-{per}: fewer than 2 clusters with ≥3 observations — skipped")

### §3d — Part A Synthesis Table

We condense the three tests × four sub-analyses into one table, a row per sub-analysis:
- **Occurrence:** χ² and Cramér's V — a larger V means the clusters differ more in whether cities get spots;
- **SUHII** and **area:** Kruskal–Wallis H and η² — a larger η² means the cluster explains more variance;
- stars: `***` p < 0.001 · `**` p < 0.01 · `*` p < 0.05 · `ns` not significant.

In [ ]:
# ── Part A Synthesis Table ────────────────────────────────────────────────────
_rows = []
for st in SPOT_TYPES:
    for per in PERIODS:
        key = (st, per)
        r_occ  = partA_occ.get(key)
        r_suh  = partA_suhii.get(key)
        r_area = partA_area.get(key)

        _rows.append({
            'Spot Type':    st.capitalize(),
            'Period':       per.capitalize(),
            # Occurrence
            'χ² (occ)':    f"{r_occ['chi2']:.2f}{sig_stars(r_occ['p'])}"  if r_occ  else '—',
            'V (occ)':      f"{r_occ['cramers_v']:.3f}"                    if r_occ  else '—',
            'p (occ)':      fmt_p(r_occ['p'])                              if r_occ  else '—',
            # SUHII
            'H (suhii)':    f"{r_suh['H']:.2f}{sig_stars(r_suh['p'])}"    if r_suh  else '—',
            'η² (suhii)':   f"{r_suh['eta2']:.3f}"                         if r_suh  else '—',
            'p (suhii)':    fmt_p(r_suh['p'])                              if r_suh  else '—',
            # Area
            'H (area)':     f"{r_area['H']:.2f}{sig_stars(r_area['p'])}"  if r_area else '—',
            'η² (area)':    f"{r_area['eta2']:.3f}"                        if r_area else '—',
            'p (area)':     fmt_p(r_area['p'])                             if r_area else '—',
        })

synth_A = pd.DataFrame(_rows)
print("=" * 100)
print("PART A SYNTHESIS — Cluster25 Association (Item 2)")
print("=" * 100)
print(synth_A.to_string(index=False))
print("\nInterpretation:")
print("  Cramér's V: small=0.10, medium=0.30, large=0.50")
print("  η²:         small=0.01, medium=0.06, large=0.14")

# ── Save synthesis table ─────────────────────────────────────────────────────
synth_A.to_csv(TABLES_DIR / 'partA_synthesis_cluster25.csv', index=False)
print(f"\n✓ Saved: tables/partA_synthesis_cluster25.csv")

# ── Heatmap of effect sizes ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (spot_type, ax) in enumerate(zip(SPOT_TYPES, axes)):
    sub = synth_A[synth_A['Spot Type'] == spot_type.capitalize()].copy()
    labels = [f"{r['Period']}" for _, r in sub.iterrows()]

    def _safe_float(v):
        try: return float(str(v).replace('***','').replace('**','').replace('*','').replace('ns',''))
        except: return 0.0

    occ_v  = [_safe_float(v) for v in sub['V (occ)']]
    suh_v  = [_safe_float(v) for v in sub['η² (suhii)']]
    area_v = [_safe_float(v) for v in sub['η² (area)']]

    matrix = np.array([occ_v, suh_v, area_v])
    im = ax.imshow(matrix, cmap='YlOrRd', vmin=0, vmax=0.5, aspect='auto')
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=11)
    ax.set_yticks([0,1,2])
    ax.set_yticklabels(['Occurrence\n(Cramér\'s V)', 'SUHII\n(η²)', 'Area\n(η²)'], fontsize=10)
    ax.set_title(f'{spot_type.capitalize()} Spots — Cluster25 Effect Sizes', fontsize=12, fontweight='bold')
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f'{matrix[i,j]:.3f}', ha='center', va='center', fontsize=11, fontweight='bold',
                    color='white' if matrix[i,j] > 0.25 else '#333333')

plt.colorbar(im, ax=axes[-1], label='Effect size', shrink=0.8)
fig.suptitle('Part A — Cluster25 Effect Size Heatmap (Day | Night × Hot | Cold)', fontsize=13, fontweight='bold')
plt.tight_layout()
for _ext in ('svg', 'pdf', 'png'):
    plt.savefig(FIGS_DIR / f'partA_effect_size_heatmap.{_ext}', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figures/partA_effect_size_heatmap.png")

<a id="nb06-s4"></a>

## §4 — Part B: Element-level Attribution (Item 1)

### Analytical Rationale

Part A shows *whether* the composite typology differentiates the cities; Part B asks *why*, by testing each
element — climate zone, geology, elevation — with the same three tests (occurrence chi-square, SUHII and
area Kruskal–Wallis), hot and cold apart. We correct the p-values with Holm's procedure within each outcome
family, across the three element tests.

### Multiple Testing Correction — Holm Procedure

With $m$ tests in a family (here $m = 3$ elements per outcome), Holm's step-down procedure controls the
family-wise error rate at $\alpha$, less conservatively than Bonferroni:

1. Sort the raw p-values: $p_{(1)} \leq p_{(2)} \leq \cdots \leq p_{(m)}$.
2. Compare each $p_{(i)}$ with $\alpha / (m - i + 1)$.
3. Reject $H_{(i)}$ while $p_{(i)} \leq \alpha / (m - i + 1)$, and stop at the first that fails.

The adjusted p-value is $p_{(i)}^{\text{Holm}} = \min\bigl(1,\; \max_{j \leq i}[(m-j+1) \cdot p_{(j)}]\bigr)$.

### Presentation Strategy

Instead of 36 separate outputs (3 elements × 3 outcomes × 4 sub-analyses), we give one table per outcome
(occurrence, SUHII, area), with the elements as rows and the statistic, effect size, raw and Holm p and
significance as columns, so that the strongest element stands out.

In [ ]:
# ── Part B: Element-level tests ──────────────────────────────────────────────
from statsmodels.stats.multitest import multipletests

partB_results = {}  # keyed by (spot_type, period, element_col, outcome)

for (st, per), df in CITY_TABLES.items():
    for elem in ELEMENT_COLS:
        r_occ  = run_chi2(df, elem, 'occurrence')
        r_suh  = run_kw(df, elem, 'suhii')
        r_area = run_kw(df, elem, 'total_area_km2_50')
        partB_results[(st, per, elem, 'occurrence')] = r_occ
        partB_results[(st, per, elem, 'suhii')]      = r_suh
        partB_results[(st, per, elem, 'area')]       = r_area

print("Part B tests computed (raw p-values).")
print("Applying Holm correction within each outcome family (3 element tests per outcome)...\n")

# ── Holm correction within each (spot_type, period, outcome) family ───────────
partB_holm = {}  # keyed by (st, per, elem, outcome) → holm-adjusted p

for st in SPOT_TYPES:
    for per in PERIODS:
        for outcome in ('occurrence', 'suhii', 'area'):
            raw_p = []
            keys  = []
            for elem in ELEMENT_COLS:
                r = partB_results.get((st, per, elem, outcome))
                if r is not None:
                    p_val = r['p']
                    raw_p.append(p_val)
                    keys.append((st, per, elem, outcome))
                else:
                    partB_holm[(st, per, elem, outcome)] = np.nan
            if len(raw_p) >= 2:
                adj = multipletests(raw_p, method='holm')[1]
                for key, p_adj in zip(keys, adj):
                    partB_holm[key] = float(p_adj)
            else:
                for key in keys:
                    partB_holm[key] = raw_p[0] if raw_p else np.nan

print("Holm correction applied ✓")
print("Results stored in partB_results and partB_holm dicts.")

In [ ]:
# ── Part B Synthesis Tables ───────────────────────────────────────────────────
def build_partB_table(spot_type, period):
    # Build Part B synthesis table for one spot_type x period combination.
    rows = []
    for elem in ELEMENT_COLS:
        r_occ  = partB_results.get((spot_type, period, elem, 'occurrence'))
        r_suh  = partB_results.get((spot_type, period, elem, 'suhii'))
        r_area = partB_results.get((spot_type, period, elem, 'area'))

        p_occ_holm  = partB_holm.get((spot_type, period, elem, 'occurrence'), np.nan)
        p_suh_holm  = partB_holm.get((spot_type, period, elem, 'suhii'),      np.nan)
        p_area_holm = partB_holm.get((spot_type, period, elem, 'area'),       np.nan)

        rows.append({
            'Element':         ELEMENT_LABELS[elem],
            # Occurrence
            'χ²':              f"{r_occ['chi2']:.2f}"       if r_occ  else '—',
            'V':               f"{r_occ['cramers_v']:.3f}"  if r_occ  else '—',
            'p(raw)_occ':      fmt_p(r_occ['p'])            if r_occ  else '—',
            'p(Holm)_occ':     fmt_p(p_occ_holm),
            'sig_occ':         sig_stars(p_occ_holm),
            # SUHII
            'H_suhii':         f"{r_suh['H']:.2f}"          if r_suh  else '—',
            'η²_suhii':        f"{r_suh['eta2']:.3f}"        if r_suh  else '—',
            'p(raw)_suhii':    fmt_p(r_suh['p'])             if r_suh  else '—',
            'p(Holm)_suhii':   fmt_p(p_suh_holm),
            'sig_suhii':       sig_stars(p_suh_holm),
            # Area
            'H_area':          f"{r_area['H']:.2f}"          if r_area else '—',
            'η²_area':         f"{r_area['eta2']:.3f}"        if r_area else '—',
            'p(raw)_area':     fmt_p(r_area['p'])             if r_area else '—',
            'p(Holm)_area':    fmt_p(p_area_holm),
            'sig_area':        sig_stars(p_area_holm),
        })
    return pd.DataFrame(rows)


# Print and save all four sub-analysis tables
for st in SPOT_TYPES:
    for per in PERIODS:
        tbl = build_partB_table(st, per)
        label = f"{st.upper()} {per.upper()}"
        print(f"\n{'='*100}")
        print(f"  PART B — {label} | Element-level Attribution")
        print(f"{'='*100}")
        display_cols = ['Element',
                        'χ²', 'V', 'p(Holm)_occ',   'sig_occ',
                        'H_suhii', 'η²_suhii', 'p(Holm)_suhii', 'sig_suhii',
                        'H_area',  'η²_area',  'p(Holm)_area',  'sig_area']
        print(tbl[display_cols].to_string(index=False))
        fname = TABLES_DIR / f'partB_{st}_{per}.csv'
        tbl.to_csv(fname, index=False)
        print(f"  ✓ Saved: {fname.name}")

In [ ]:
# ── Part B: Element effect-size comparison plot ───────────────────────────────
# 4-panel grid: hot-day | hot-night | cold-day | cold-night
# Each panel: grouped bar chart with Cramér's V (occurrence) and η² (suhii/area)

fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=False)
ax_map = {('hot','day'):axes[0,0], ('hot','night'):axes[0,1],
          ('cold','day'):axes[1,0], ('cold','night'):axes[1,1]}

for (st, per), ax in ax_map.items():
    elem_short = [ELEMENT_LABELS[e].replace(' Zone','').replace(' Belt','') for e in ELEMENT_COLS]
    x = np.arange(len(ELEMENT_COLS))
    w = 0.26

    v_vals  = []; h_suh_vals = []; h_area_vals = []
    sig_occ = []; sig_suh = []; sig_area = []

    for elem in ELEMENT_COLS:
        r_occ  = partB_results.get((st, per, elem, 'occurrence'))
        r_suh  = partB_results.get((st, per, elem, 'suhii'))
        r_area = partB_results.get((st, per, elem, 'area'))
        p_occ_h  = partB_holm.get((st, per, elem, 'occurrence'), 1)
        p_suh_h  = partB_holm.get((st, per, elem, 'suhii'),      1)
        p_area_h = partB_holm.get((st, per, elem, 'area'),       1)

        v_vals.append(r_occ['cramers_v']  if r_occ  else 0)
        h_suh_vals.append(r_suh['eta2']  if r_suh  else 0)
        h_area_vals.append(r_area['eta2'] if r_area else 0)
        sig_occ.append(sig_stars(p_occ_h))
        sig_suh.append(sig_stars(p_suh_h))
        sig_area.append(sig_stars(p_area_h))

    bars1 = ax.bar(x - w, v_vals,       w, label="Occurrence (V)",  color='#4e79a7', edgecolor='white')
    bars2 = ax.bar(x,     h_suh_vals,   w, label="SUHII (η²)",      color='#f28e2b', edgecolor='white')
    bars3 = ax.bar(x + w, h_area_vals,  w, label="Area (η²)",       color='#59a14f', edgecolor='white')

    # Significance annotations
    for i, (b1, b2, b3) in enumerate(zip(bars1, bars2, bars3)):
        for bar, sig in [(b1, sig_occ[i]), (b2, sig_suh[i]), (b3, sig_area[i])]:
            if sig not in ('', 'ns'):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        sig, ha='center', va='bottom', fontsize=9, color='#333333')

    ax.set_xticks(x); ax.set_xticklabels(elem_short, fontsize=10)
    ax.set_ylabel('Effect size', fontsize=10)
    ax.set_title(f'{st.capitalize()} spots — {per.capitalize()}', fontsize=11, fontweight='bold')
    ax.axhline(0.06, ls='--', lw=0.7, color='grey', alpha=0.6, label='Medium η²/V = 0.06')
    ax.set_ylim(0, max(max(v_vals+h_suh_vals+h_area_vals, default=0)*1.4, 0.2))
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Part B — Element-level Effect Sizes (Cramér\'s V and η²)\n'
             'Holm-corrected significance: * p<0.05  ** p<0.01  *** p<0.001',
             fontsize=13, fontweight='bold')
plt.tight_layout()
for _ext in ('svg', 'pdf', 'png'):
    plt.savefig(FIGS_DIR / f'partB_element_effect_sizes.{_ext}', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figures/partB_element_effect_sizes.png")

<a id="nb06-s5"></a>

## §5 — Synthesis: Linking Cluster-level and Element-level Evidence

### Analytical Rationale

Part A tests the typology as a whole and Part B its elements. The synthesis asks: *which element drives the
cluster differences, and does the composite typology explain more than any single element?* In three steps:

| Step | Method | Question answered |
|---|---|---|
| **S1** | Cramér's V between Cluster25 and each element class | How much does each element class overlap with the cluster partition? |
| **S2** | Side-by-side comparison of effect sizes (Cluster25 vs best element) | Does the cluster composite out-explain individual elements? |
| **S3** | Cluster thermal profile heatmap (median SUHII and area per cluster) | What is the thermal signature of each physiographic cluster? |

---

### S1 Rationale — Cramér's V for Cluster Composition

The typology combines climate, geology and elevation. A high Cramér's V between Cluster25 and one element
(say `climate_class`) means that element almost fixes the cluster, so the cluster adds little beyond it; low
values for all three mean the cluster is a genuine composite, adding value by combining dimensions.

In [ ]:
# ── S1: Cramér's V — Cluster25 vs each element class ─────────────────────────
# (Only uses the 55 cities in GDF; cluster25=-1 cities excluded)
city_land_valid = city_landscape[city_landscape['cluster25'] != -1].copy()

s1_rows = []
for elem in ELEMENT_COLS:
    sub = city_land_valid[['cluster25', elem]].dropna()
    ct  = pd.crosstab(sub['cluster25'], sub[elem])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        continue
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * min(ct.shape[0]-1, ct.shape[1]-1)))
    s1_rows.append({
        'Element Class':    ELEMENT_LABELS[elem],
        'n_element_cats':   sub[elem].nunique(),
        'n_clusters':       sub['cluster25'].nunique(),
        'chi2':             round(chi2, 2),
        'p':                fmt_p(p),
        'sig':              sig_stars(p),
        "Cramér's V":       round(v, 3),
        'Interpretation':   ('Strong overlap' if v > 0.5 else
                             'Moderate overlap' if v > 0.30 else
                             'Weak overlap'),
    })

s1_df = pd.DataFrame(s1_rows).sort_values("Cramér's V", ascending=False)

print("=" * 70)
print("S1 — Cramér's V: Cluster25 vs Individual Element Classes")
print("=" * 70)
print(s1_df.to_string(index=False))
print(f"\nInterpretation:")
print(f"  High V → element strongly determines cluster membership")
print(f"  Low V  → cluster captures dimensions beyond this element")
s1_df.to_csv(TABLES_DIR / 'synthesis_S1_cramers_v.csv', index=False)
print(f"\n✓ Saved: tables/synthesis_S1_cramers_v.csv")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d73027' if v > 0.5 else '#fc8d59' if v > 0.3 else '#91bfdb'
          for v in s1_df["Cramér's V"]]
ax.barh(s1_df['Element Class'], s1_df["Cramér's V"], color=colors, edgecolor='white')
ax.axvline(0.5, ls='--', lw=1, color='#d73027', alpha=0.7, label='Strong overlap (V=0.50)')
ax.axvline(0.3, ls='--', lw=1, color='#fc8d59', alpha=0.7, label='Moderate (V=0.30)')
ax.set_xlabel("Cramér's V", fontsize=11)
ax.set_title("S1 — Cluster25 Overlap with Each Landscape Element Class", fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
for _ext in ('svg', 'pdf', 'png'):
    plt.savefig(FIGS_DIR / f'synthesis_S1_cramers_v.{_ext}', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figures/synthesis_S1_cramers_v.png")

### S2 — Comparative Explanatory Power: Cluster25 vs Best Element

Does the composite typology explain more variance in SUHII and area than any single element? We compare the
Kruskal–Wallis η² of Cluster25 (Part A) with the highest η² of an element (Part B), per sub-analysis. If
$\eta^2_{\text{Cluster25}} > \eta^2_{\text{best element}}$, the composite adds explanatory value; if the two
are close, the best element alone suffices.

In [ ]:
# ── S2: Comparative explanatory power ────────────────────────────────────────
s2_rows = []
for st in SPOT_TYPES:
    for per in PERIODS:
        key = (st, per)
        for outcome, label, a_result_dict in [
            ('suhii', 'SUHII',       partA_suhii),
            ('area',  'Total Area',  partA_area),
        ]:
            r_clust = a_result_dict.get(key)
            eta2_clust = r_clust['eta2'] if r_clust else np.nan

            # Best element from Part B for this outcome
            elem_eta2 = {}
            for elem in ELEMENT_COLS:
                r = partB_results.get((st, per, elem, outcome))
                if r:
                    elem_eta2[ELEMENT_LABELS[elem]] = r['eta2']
            if elem_eta2:
                best_elem  = max(elem_eta2, key=elem_eta2.get)
                best_eta2  = elem_eta2[best_elem]
                delta      = eta2_clust - best_eta2
                added_val  = 'Yes' if delta > 0.01 else ('Similar' if abs(delta) <= 0.01 else 'No')
            else:
                best_elem, best_eta2, delta, added_val = '—', np.nan, np.nan, '—'

            s2_rows.append({
                'Spot Type':          st.capitalize(),
                'Period':             per.capitalize(),
                'Outcome':            label,
                'Cluster25 η²':       round(eta2_clust, 4) if not np.isnan(eta2_clust) else '—',
                'Best element':       best_elem,
                'Best element η²':    round(best_eta2, 4)  if not np.isnan(best_eta2)  else '—',
                'Δη² (clust-elem)':   round(delta, 4)      if not np.isnan(delta)       else '—',
                'Cluster adds value': added_val,
            })

s2_df = pd.DataFrame(s2_rows)
print("=" * 100)
print("S2 — Comparative Explanatory Power: Cluster25 vs Best Individual Element")
print("=" * 100)
print(s2_df.to_string(index=False))
s2_df.to_csv(TABLES_DIR / 'synthesis_S2_comparative_power.csv', index=False)
print(f"\n✓ Saved: tables/synthesis_S2_comparative_power.csv")

### S3 — Cluster Thermal Profile Heatmap

We map the median SUHII (°C) and median total spot area (km²) of each active cluster, for hot and cold spots,
day and night: the **thermal fingerprint** of each physiographic cluster. In the SUHII map red is a strong
positive anomaly and blue cooling; in the area map darker is a larger extent.

In [ ]:
# ── S3: Cluster thermal profile heatmap ──────────────────────────────────────
def cluster_profile(spot_type, outcome_col, outcome_label, cmap, vcenter=None):
    # Build cluster x period median matrix for heatmap.
    rows = {}
    for per in PERIODS:
        df = CITY_TABLES[(spot_type, per)]
        sub = df[df['occurrence'] == 1][['cluster25', outcome_col]].dropna()
        sub = sub[sub['cluster25'] != -1]
        rows[per.capitalize()] = sub.groupby('cluster25')[outcome_col].median()
    matrix = pd.DataFrame(rows).sort_index()
    return matrix

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, spot_type in enumerate(SPOT_TYPES):
    for col_idx, (outcome_col, outcome_label, cmap, vcenter) in enumerate([
        ('suhii',              'Median SUHII (°C)',             'RdBu_r', 0),
        ('total_area_km2_90',  'Median Total Area L90 (km²)',   'YlOrRd',  None),
    ]):
        ax = axes[row_idx, col_idx]
        mat = cluster_profile(spot_type, outcome_col, outcome_label, cmap, vcenter)

        if mat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        if vcenter is not None:
            from matplotlib.colors import TwoSlopeNorm
            vmin, vmax = mat.values.min(), mat.values.max()
            if vmin < vcenter < vmax:
                norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
            else:
                norm = None
        else:
            norm = None

        im = sns.heatmap(
            mat, ax=ax, cmap=cmap, norm=norm,
            annot=True, fmt='.2f', annot_kws={'size': 9},
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': outcome_label, 'shrink': 0.8}
        )
        ax.set_xlabel('Period', fontsize=10)
        ax.set_ylabel('Cluster25', fontsize=10)
        ax.set_title(f'{spot_type.capitalize()} spots — {outcome_label}',
                     fontsize=11, fontweight='bold')

fig.suptitle('S3 — Cluster25 Thermal Profile Heatmap\n'
             '(Median values for cities with occurrence = 1)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
for _ext in ('svg', 'pdf', 'png'):
    plt.savefig(FIGS_DIR / f'synthesis_S3_cluster_profile_heatmap.{_ext}', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figures/synthesis_S3_cluster_profile_heatmap.png")

### S4 — Final Narrative Synthesis Table

We cross-tabulate the conclusions for the results text, one row per sub-analysis (4 × 2 spot types):
- **Cluster sig.:** is Cluster25 a significant discriminator for the outcome (Part A)?
- **Driving element:** which element has the largest effect after Holm correction (Part B)?
- **Element sig.:** is that element significant on its own?
- **Cluster advantage:** does Cluster25 explain more than the driving element alone?

In [ ]:
# ── S4: Final narrative synthesis table ──────────────────────────────────────
def _sig_str(p, stars):
    return f"Yes ({stars})" if stars not in ('ns', '') else "No (ns)"

s4_rows = []
for st in SPOT_TYPES:
    for per in PERIODS:
        key = (st, per)
        # Part A results
        r_occ_A   = partA_occ.get(key)
        r_suh_A   = partA_suhii.get(key)
        r_area_A  = partA_area.get(key)

        # Best Part B element per outcome (Holm-corrected)
        def _best_elem(outcome):
            eff = {e: partB_results[(st,per,e,outcome)]['eta2']
                   if outcome != 'occurrence'
                   else partB_results[(st,per,e,outcome)]['cramers_v']
                   for e in ELEMENT_COLS
                   if partB_results.get((st,per,e,outcome))}
            if not eff: return '—', '—', 'ns'
            best = max(eff, key=eff.get)
            p_h  = partB_holm.get((st, per, best, outcome), 1.0)
            return ELEMENT_LABELS[best], f"{eff[best]:.3f}", sig_stars(p_h)

        b_occ_e,  b_occ_v,  b_occ_s  = _best_elem('occurrence')
        b_suh_e,  b_suh_v,  b_suh_s  = _best_elem('suhii')
        b_area_e, b_area_v, b_area_s = _best_elem('area')

        s4_rows.append({
            'Spot Type': st.capitalize(),
            'Period':    per.capitalize(),
            # Occurrence
            'Occ: Cluster25 sig.':     _sig_str(r_occ_A['p'], sig_stars(r_occ_A['p'])) if r_occ_A  else '—',
            'Occ: Driving element':    b_occ_e,
            'Occ: Element sig.':       b_occ_s,
            # SUHII
            'SUHII: Cluster25 sig.':   _sig_str(r_suh_A['p'], sig_stars(r_suh_A['p'])) if r_suh_A  else '—',
            'SUHII: Driving element':  b_suh_e,
            'SUHII: Element sig.':     b_suh_s,
            # Area
            'Area: Cluster25 sig.':    _sig_str(r_area_A['p'], sig_stars(r_area_A['p'])) if r_area_A else '—',
            'Area: Driving element':   b_area_e,
            'Area: Element sig.':      b_area_s,
        })

s4_df = pd.DataFrame(s4_rows)
print("=" * 120)
print("S4 — FINAL SYNTHESIS TABLE")
print("=" * 120)
print(s4_df.to_string(index=False))
s4_df.to_csv(TABLES_DIR / 'synthesis_S4_narrative.csv', index=False)
print(f"\n✓ Saved: tables/synthesis_S4_narrative.csv")
print("\nAll synthesis tables saved to:", TABLES_DIR)

<a id="nb06-s6"></a>

## §6 — Appendix: Statistical Charts

We draw the supplementary charts behind the synthesis:

| Figure | Content | Purpose |
|---|---|---|
| App-1 | Distribution boxplots: SUHII and Area by Cluster25 | Show raw distributional evidence underlying Part A KW tests |
| App-2 | Dunn post-hoc p-value heatmaps (where KW significant) | Show which specific cluster pairs drive significant differences |
| App-3 | Element-class boxplots (SUHII and Area by each element) | Raw distributional evidence underlying Part B tests |

In [ ]:
# ── App-1: Distribution boxplots by Cluster25 ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for row_idx, (outcome_col, outcome_label, ylabel) in enumerate([
    ('suhii',             'SUHII (°C)',             'SUHII (°C)'),
    ('total_area_km2_90', 'Total Area L90 (km²)',    'Area km²'),
]):
    for col_idx, spot_type in enumerate(SPOT_TYPES):
        ax = axes[row_idx, col_idx]
        frames = []
        for per in PERIODS:
            df = CITY_TABLES[(spot_type, per)]
            sub = df[df['occurrence'] == 1][['cluster25', outcome_col, 'period']].dropna()
            frames.append(sub)
        if not frames: continue
        combined = pd.concat(frames, ignore_index=True)
        combined = combined[combined['cluster25'] != -1]
        combined['cluster25'] = combined['cluster25'].astype(int)

        # Order clusters by median
        order = (combined.groupby('cluster25')[outcome_col]
                         .median().sort_values(ascending=False).index.tolist())

        sns.boxplot(data=combined, x='cluster25', y=outcome_col, hue='period',
                    order=order, palette={'day': '#FF8C00', 'night': '#1565C0'},
                    width=0.6, flierprops={'markersize': 3}, ax=ax)

        # Annotate KW result
        r = partA_suhii.get((spot_type, 'day')) if outcome_col == 'suhii' else partA_area.get((spot_type, 'day'))
        if r:
            ax.text(0.98, 0.97, f"KW H={r['H']:.1f} {sig_stars(r['p'])}",
                    transform=ax.transAxes, ha='right', va='top', fontsize=9,
                    bbox={'fc': 'white', 'ec': 'grey', 'alpha': 0.8, 'pad': 2})

        ax.set_xlabel('Cluster25', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f'{spot_type.capitalize()} spots — {outcome_label}',
                     fontsize=11, fontweight='bold')
        ax.tick_params(axis='x', labelsize=8)
        if ax.get_legend(): ax.get_legend().set_title('Period', prop={'size': 9})
        ax.grid(axis='y', alpha=0.3)

fig.suptitle('Appendix Fig 1 — Cluster25 Distributions (SUHII and Total Area L90)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
for _ext in ('svg', 'pdf', 'png'):
    plt.savefig(FIGS_DIR / f'appendix_A1_cluster_boxplots.{_ext}', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figures/appendix_A1_cluster_boxplots.png")

In [ ]:
# ── App-2: Dunn post-hoc heatmaps (significant KW tests only) ────────────────
for outcome_label, result_dict in [('SUHII', partA_suhii), ('Area', partA_area)]:
    for (st, per), result in result_dict.items():
        if result is None or result['p'] >= ALPHA or result['dunn'] is None:
            continue
        dunn = result['dunn']
        fig, ax = plt.subplots(figsize=(max(6, len(dunn)*0.6), max(5, len(dunn)*0.5)))
        mask = np.eye(len(dunn), dtype=bool)
        # Annotate significant pairs
        annot = dunn.copy().applymap(lambda x: f'{x:.3f}' if x < 0.05 else '')
        sns.heatmap(dunn, ax=ax, cmap='RdYlGn_r', vmin=0, vmax=0.1,
                    annot=annot, fmt='', annot_kws={'size': 7},
                    linewidths=0.3, linecolor='white', mask=mask,
                    cbar_kws={'label': 'Holm-adjusted p-value'})
        ax.set_title(f'Dunn Post-hoc — {outcome_label} × Cluster25 | {st.capitalize()} {per.capitalize()}\n'
                     f'(Holm-corrected; green = p < 0.05)',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Cluster25'); ax.set_ylabel('Cluster25')
        plt.tight_layout()
        _stem33 = f'appendix_A2_dunn_{outcome_label.lower()}_{st}_{per}'
        for _ext in ('svg', 'pdf', 'png'):
            plt.savefig(FIGS_DIR / f'{_stem33}.{_ext}', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✓ Saved: {fname.name}")

In [ ]:
# ── App-3: Element-class boxplots (SUHII and Area by each landscape element) ──
fig, axes = plt.subplots(len(ELEMENT_COLS), 2, figsize=(16, 5*len(ELEMENT_COLS)))

for row_idx, elem in enumerate(ELEMENT_COLS):
    for col_idx, (outcome_col, outcome_label) in enumerate([
        ('suhii',             'SUHII (°C)'),
        ('total_area_km2_90', 'Total Area L90 (km²)'),
    ]):
        ax = axes[row_idx, col_idx]
        frames = []
        for st in SPOT_TYPES:
            for per in PERIODS:
                df = CITY_TABLES[(st, per)]
                sub = df[df['occurrence'] == 1][[elem, outcome_col, 'spot_type', 'period']].dropna()
                frames.append(sub)
        if not frames: continue
        combined = pd.concat(frames, ignore_index=True)

        # Order categories by median value
        order = (combined.groupby(elem)[outcome_col]
                         .median().sort_values(ascending=False).index.tolist())
        sns.boxplot(data=combined, x=elem, y=outcome_col, hue='spot_type',
                    order=order, palette={'hot': '#FF2400', 'cold': '#00BFFF'},
                    width=0.6, flierprops={'markersize': 3}, ax=ax)
        ax.set_xlabel(ELEMENT_LABELS[elem], fontsize=9)
        ax.set_ylabel(outcome_label, fontsize=9)
        ax.set_title(f'{ELEMENT_LABELS[elem]} — {outcome_label}',
                     fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=20, labelsize=8)
        if ax.get_legend(): ax.get_legend().set_title('Spot type', prop={'size': 8})
        ax.axhline(0, ls='--', lw=0.8, color='grey', alpha=0.5)
        ax.grid(axis='y', alpha=0.3)

fig.suptitle('Appendix Fig 3 — Physiographic Element Distributions (SUHII and Total Area L90; 3 elements)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
for _ext in ('svg', 'pdf', 'png'):
    plt.savefig(FIGS_DIR / f'appendix_A3_element_boxplots.{_ext}', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figures/appendix_A3_element_boxplots.png")
print("\n─── NB06 Complete ───")
print(f"All tables → {TABLES_DIR}")
print(f"All figures → {FIGS_DIR}")

<a id="nb06-s7"></a>

## §7 — Master Synthesis Table & Forest Plot

### §7a — Master Synthesis Table

We bring all Part A and Part B results into one table, a row per spot type × period × outcome: for Cluster25
the statistic, effect size (V or η²), p and significance; for the best element its name, effect size,
Holm p and significance; and Δ, the Cluster25 effect minus the best element's (the typology's added value).

### §7b — Forest Plot

We plot every effect size (V for occurrence, η² for SUHII and area) of Part A and Part B, grouped by outcome:
filled symbols are significant (p < 0.05), open symbols not.

In [ ]:
# ── §7a: Master Synthesis Table ─────────────────────────────────────────────
_outcomes   = ['occurrence', 'suhii', 'area']
_out_labels = {'occurrence': 'Occurrence', 'suhii': 'SUHII', 'area': 'Area (L50)'}
_part_a_src = {'occurrence': partA_occ, 'suhii': partA_suhii, 'area': partA_area}
_eff_label  = {'occurrence': 'V (Cramér)',  'suhii': 'η²', 'area': 'η²'}

master_rows = []
for st in SPOT_TYPES:
    for per in PERIODS:
        for out in _outcomes:
            key = (st, per)
            rA  = _part_a_src[out].get(key)

            # Part A stats
            if rA:
                if out == 'occurrence':
                    a_stat   = round(rA['chi2'],        3)
                    a_eff    = round(rA['cramers_v'],   3)
                    a_method = rA.get('method', 'chi2')
                else:
                    a_stat   = round(rA['H'],    3)
                    a_eff    = round(rA['eta2'],  3)
                    a_method = 'KW'
                a_p   = rA['p']
                a_sig = sig_stars(a_p)
            else:
                a_stat = a_eff = a_p = np.nan
                a_sig = '—'; a_method = '—'

            # Best Part B element
            b_effs = {}
            for elem in ELEMENT_COLS:
                rB = partB_results.get((st, per, elem, out))
                if rB:
                    b_effs[elem] = rB['cramers_v'] if out == 'occurrence' else rB['eta2']
            best_elem     = max(b_effs, key=b_effs.get) if b_effs else None
            b_eff         = round(b_effs[best_elem], 3)  if best_elem else np.nan
            b_p_holm      = partB_holm.get((st, per, best_elem, out), np.nan) if best_elem else np.nan
            b_sig         = sig_stars(b_p_holm)          if best_elem else '—'
            best_elem_lbl = ELEMENT_LABELS.get(best_elem, '—') if best_elem else '—'

            delta = round(float(a_eff) - float(b_eff), 3) if not np.isnan(a_eff) and not np.isnan(b_eff) else np.nan

            master_rows.append({
                'Spot Type':          st.capitalize(),
                'Period':             per.capitalize(),
                'Outcome':            _out_labels[out],
                'A: Method':          a_method,
                f'A: {_eff_label[out]}': a_eff,
                'A: p':               fmt_p(a_p),
                'A: sig':             a_sig,
                'B: Best element':    best_elem_lbl,
                f'B: {_eff_label[out]}': b_eff,
                'B: p(Holm)':         fmt_p(b_p_holm),
                'B: sig':             b_sig,
                'Δ effect size':      delta,
            })

master_df = pd.DataFrame(master_rows)
print("=" * 130)
print("§7a — MASTER SYNTHESIS TABLE")
print("=" * 130)
print(master_df.to_string(index=False))
master_df.to_csv(TABLES_DIR / 'master_synthesis.csv', index=False)
print(f"\n✓ Saved: tables/master_synthesis.csv  ({len(master_df)} rows × {len(master_df.columns)} cols)")
print()
print("Summary — significant results (p < 0.05):")
sig_mask = master_df['A: sig'].isin(['*','**','***'])
print(master_df[sig_mask][['Spot Type','Period','Outcome','A: sig',
                            'B: Best element','B: sig','Δ effect size']].to_string(index=False))


In [ ]:
# ── §7b: Forest Plot — all effect sizes ──────────────────────────────────────
from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.lines   import Line2D

_FS_THRESH = {
    'occurrence': {'small': 0.10, 'medium': 0.30, 'large': 0.50,
                   'floor': 0.60, 'eff_lbl': "Cramér's V"},
    'suhii':      {'small': 0.01, 'medium': 0.06, 'large': 0.14,
                   'floor': 0.22, 'eff_lbl': 'η²'},
    'area':       {'small': 0.01, 'medium': 0.06, 'large': 0.14,
                   'floor': 0.22, 'eff_lbl': 'η² (L50)'},
}

_ZONE_CLR = {'negligible': '#FFAAAA', 'small': '#FFCC99',
             'medium': '#FFEE99',      'large': '#AADDAA'}
_ZONE_A = 0.42

# Faint fill colours — significance shown via outline only
_clust_fill = '#A8D8A8'
_elem_fills = {'climate_class':   '#F4A8A8',
               'geology_class':   '#A8A8F4',
               'elevation_class': '#F4CCA8'}
_elem_marks = {'climate_class': 'D', 'geology_class': 's', 'elevation_class': '^'}

# Significance outlines
_ec_sig = '#CC1100'   # Mars red — significant
_ec_ns  = '#111111'   # near-black — not significant
_lw_sig, _lw_ns = 2.0, 0.9

_fs_base, _fs_title = 10, 11


def _collect_effs(out):
    vals = []
    for st in SPOT_TYPES:
        for per in PERIODS:
            rA = _part_a_src[out].get((st, per))
            if rA:
                vals.append(rA['cramers_v'] if out == 'occurrence' else rA['eta2'])
            for elem in ELEMENT_COLS:
                rB = partB_results.get((st, per, elem, out))
                if rB:
                    vals.append(rB['cramers_v'] if out == 'occurrence' else rB['eta2'])
    return vals


# A4 landscape
fig_f, axes_f = plt.subplots(1, 3, figsize=(11.69, 8.27),
                              gridspec_kw={'wspace': 0.46})

for ax_f, out in zip(axes_f, ['occurrence', 'suhii', 'area']):
    thresh  = _FS_THRESH[out]
    s, m, l = thresh['small'], thresh['medium'], thresh['large']
    eff_lbl = thresh['eff_lbl']
    all_effs = _collect_effs(out)
    xlim = max(thresh['floor'], max(all_effs) * 1.15) if all_effs else thresh['floor']

    ax_f.set_title(_out_labels[out], fontsize=_fs_title, fontweight='bold', pad=8)
    ax_f.set_xlabel(eff_lbl, fontsize=_fs_base)
    ax_f.set_xlim(-0.01, xlim)

    ax_f.axvspan(-0.01, s,    color=_ZONE_CLR['negligible'], alpha=_ZONE_A, zorder=0)
    ax_f.axvspan( s,    m,    color=_ZONE_CLR['small'],      alpha=_ZONE_A, zorder=0)
    ax_f.axvspan( m,    l,    color=_ZONE_CLR['medium'],     alpha=_ZONE_A, zorder=0)
    ax_f.axvspan( l,    xlim, color=_ZONE_CLR['large'],      alpha=_ZONE_A, zorder=0)

    for ref, ls in ((s, '--'), (m, ':'), (l, '-.')):
        ax_f.axvline(ref, color='#888888', lw=0.7, ls=ls, zorder=1)
    ax_f.axvline(0, color='#CCCCCC', lw=0.7, zorder=1)

    _y_labels, _y_pos, y_cur = [], [], 0
    for st in SPOT_TYPES:
        for per in PERIODS:
            _y_labels.append(f'{st.capitalize()} / {per.capitalize()}')
            _y_pos.append(y_cur)

            rA = _part_a_src[out].get((st, per))
            if rA:
                eff_A = rA['cramers_v'] if out == 'occurrence' else rA['eta2']
                sig_A = rA['p'] < ALPHA
                ax_f.scatter(eff_A, y_cur,
                             color=_clust_fill, s=110, marker='o', zorder=5,
                             edgecolors=_ec_sig if sig_A else _ec_ns,
                             linewidths=_lw_sig if sig_A else _lw_ns)
                if sig_A:
                    ax_f.text(eff_A + xlim * 0.015, y_cur + 0.20,
                              sig_stars(rA['p']), fontsize=8,
                              color=_ec_sig, va='bottom', fontweight='bold')

            offsets = {'climate_class': -0.22, 'geology_class': 0.0,
                       'elevation_class': 0.22}
            for elem in ELEMENT_COLS:
                rB = partB_results.get((st, per, elem, out))
                if rB:
                    eff_B  = rB['cramers_v'] if out == 'occurrence' else rB['eta2']
                    p_holm = partB_holm.get((st, per, elem, out), 1.0)
                    sig_B  = p_holm < ALPHA
                    ax_f.scatter(eff_B, y_cur + offsets[elem],
                                 color=_elem_fills[elem],
                                 s=60, marker=_elem_marks[elem], zorder=4,
                                 edgecolors=_ec_sig if sig_B else _ec_ns,
                                 linewidths=_lw_sig if sig_B else _lw_ns)
            y_cur += 1

    ax_f.set_yticks(_y_pos)
    ax_f.set_yticklabels(_y_labels, fontsize=_fs_base)
    ax_f.invert_yaxis()
    ax_f.grid(True, axis='x', alpha=0.18, ls='--', zorder=2)
    ax_f.tick_params(axis='x', labelsize=_fs_base - 1)
    ax_f.spines['top'].set_visible(False)
    ax_f.spines['right'].set_visible(False)

fig_f.suptitle('Forest Plot: Effect Sizes Across Physiographic Predictors and Outcomes',
               fontsize=_fs_title + 1, fontweight='bold')

# Reserve bottom 30% for the legend panel
fig_f.subplots_adjust(top=0.90, bottom=0.30, left=0.08, right=0.98)

# ── Outer bounding box (rendered behind legends, zorder=0) ───────────────────
_outer = FancyBboxPatch(
    (0.01, 0.01), 0.97, 0.26,
    boxstyle='square,pad=0',
    facecolor='#fafafa', edgecolor='#aaaaaa', linewidth=1.2,
    transform=fig_f.transFigure, zorder=0, clip_on=False
)
fig_f.add_artist(_outer)

# Two vertical separators dividing the box into three sections
for _sx in (0.34, 0.67):
    fig_f.add_artist(
        Line2D([_sx, _sx], [0.015, 0.265],
               transform=fig_f.transFigure,
               color='#cccccc', lw=1.0, zorder=1, clip_on=False)
    )

# ── Legend group 1 — Effect Size Zone ────────────────────────────────────────
_zone_h = [
    Patch(facecolor=_ZONE_CLR['negligible'], alpha=_ZONE_A+0.2,
          edgecolor='#888', lw=0.6, label='Negligible'),
    Patch(facecolor=_ZONE_CLR['small'],      alpha=_ZONE_A+0.2,
          edgecolor='#888', lw=0.6, label='Small'),
    Patch(facecolor=_ZONE_CLR['medium'],     alpha=_ZONE_A+0.2,
          edgecolor='#888', lw=0.6, label='Medium'),
    Patch(facecolor=_ZONE_CLR['large'],      alpha=_ZONE_A+0.2,
          edgecolor='#888', lw=0.6, label='Large'),
]
_lkw = dict(fontsize=_fs_base - 0.5, frameon=False,
            handlelength=1.2, handletextpad=0.5, borderpad=0.0,
            columnspacing=0.9, bbox_transform=fig_f.transFigure)

leg1 = fig_f.legend(handles=_zone_h, title='Effect Size Zone',
                    title_fontsize=_fs_base,
                    loc='lower left', bbox_to_anchor=(0.03, 0.03),
                    ncol=2, **_lkw)
leg1.get_title().set_fontweight('bold')

# ── Legend group 2 — Predictor ────────────────────────────────────────────────
# Neutral grey outline (#666666): shows marker shape without implying significance
_neutral_ec = '#666666'
_pred_h = [
    Line2D([0],[0], marker='o', ls='none', color=_clust_fill,
           markeredgecolor=_neutral_ec, markeredgewidth=1.0,
           markersize=9, label='Cluster25'),
    Line2D([0],[0], marker='D', ls='none', color=_elem_fills['climate_class'],
           markeredgecolor=_neutral_ec, markeredgewidth=1.0,
           markersize=7, label='Climate Zone'),
    Line2D([0],[0], marker='s', ls='none', color=_elem_fills['geology_class'],
           markeredgecolor=_neutral_ec, markeredgewidth=1.0,
           markersize=7, label='Geology'),
    Line2D([0],[0], marker='^', ls='none', color=_elem_fills['elevation_class'],
           markeredgecolor=_neutral_ec, markeredgewidth=1.0,
           markersize=8, label='Elevation Belt'),
]
leg2 = fig_f.legend(handles=_pred_h, title='Predictor',
                    title_fontsize=_fs_base,
                    loc='lower left', bbox_to_anchor=(0.36, 0.03),
                    ncol=2, **_lkw)
leg2.get_title().set_fontweight('bold')

# ── Legend group 3 — Significance ────────────────────────────────────────────
# Shows both fill colour AND outline so reader knows what to look for on the plot
_sig_h = [
    Line2D([0],[0], marker='o', ls='none', color='#AAAAAA',
           markeredgecolor=_ec_sig, markeredgewidth=_lw_sig,
           markersize=10, label='Significant'),
    Line2D([0],[0], marker='o', ls='none', color='#AAAAAA',
           markeredgecolor=_ec_ns, markeredgewidth=_lw_ns,
           markersize=10, label='Not significant'),
]
leg3 = fig_f.legend(handles=_sig_h, title='Significance (p < 0.05)',
                    title_fontsize=_fs_base,
                    loc='lower left', bbox_to_anchor=(0.69, 0.03),
                    ncol=1, **_lkw)
leg3.get_title().set_fontweight('bold')

for _ext in ('svg', 'pdf', 'png'):
    fig_f.savefig(FIGS_DIR / f'forest_plot_all_effects.{_ext}',
                  bbox_inches='tight', dpi=500)
plt.show()
print("✓ Forest plot saved: forest_plot_all_effects.svg / .pdf / .png")


<a id="nb06-s8"></a>

## §8 — Notebook Summary and Key Outputs

### Purpose

We tested whether the Cluster25 typology, and its three elements (climate zone, geology, elevation belt),
explain how the surface hot and cold spots of Nigerian cities vary. This is the last analytical notebook of
NB01–NB06 and gives the statistical evidence on the physiographic controls.

---

### Analytical Framework Adopted

We first planned hierarchical regression models. With a median of only 3–5 cities per cluster, regression
inference is unreliable, so we use non-parametric association tests, which assume no distribution and hold
at small n:

| Outcome | Test | Effect size | Post-hoc |
|---------|------|-------------|----------|
| Occurrence (binary) | Monte Carlo permutation χ² | Cramér's V | — |
| SUHII (°C) | Kruskal-Wallis | η² | Dunn + Holm |
| Total hotspot area (km²) | Kruskal-Wallis | η² | Dunn + Holm |
We test two kinds of grouping: **Part A**, the Cluster25 composite (16 active classes, four sub-analyses:
hot / cold × day / night); **Part B**, each element (climate zone, geology, elevation belt), with Holm's
correction within each outcome family.

---

### Outputs Produced

| File | Location | Contents |
|------|----------|----------|
| `partA_synthesis_cluster25.csv` | `tables/` | χ²/KW results for all 4 sub-analyses × 3 outcomes |
| `partB_synthesis_{spot}_{period}.csv` | `tables/` | Element-level results per sub-analysis |
| `synthesis_S1_cramers_v.csv` | `tables/` | Cramér's V — Cluster25 vs each element class |
| `synthesis_S2_comparative_power.csv` | `tables/` | Δη², the comparative explanatory power |
| `synthesis_S4_narrative.csv` | `tables/` | Compact cross-tabulation for Results section prose |
| `master_synthesis.csv` | `tables/` | Master table: all Part A + Part B results, one row per combination |
| `partA_effect_size_heatmap.svg/.pdf/.png` | `figures/` | Effect-size heatmap — Cluster25 |
| `partB_element_effect_sizes.svg/.pdf/.png` | `figures/` | Element comparison — 4 sub-analyses |
| `synthesis_S1_cramers_v.svg/.pdf/.png` | `figures/` | S1 Cramér's V comparison |
| `synthesis_S3_cluster_profile_heatmap.svg/.pdf/.png` | `figures/` | Cluster thermal profile (SUHII + area) |
| `appendix_A1_cluster_boxplots.svg/.pdf/.png` | `figures/` | Cluster25 distribution boxplots |
| `appendix_A2_dunn_*.svg/.pdf/.png` | `figures/` | Dunn post-hoc heatmaps |
| `appendix_A3_element_boxplots.svg/.pdf/.png` | `figures/` | Element-class boxplots |
| `forest_plot_all_effects.svg/.pdf/.png` | `figures/` | Master forest plot — all effect sizes, A4, 500 dpi |

---